In [1]:
from pathlib import Path
import os
import sys

# 定位项目根目录，并切换工作目录、加入 Python 搜索路径
REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "jupyter_notebooks":
    REPO_ROOT = REPO_ROOT.parent

os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# #############################################################################
# 参数总览（按功能分类，方便定位与设置）
# -----------------------------------------------------------------------------
#   一、运行环境          —— 计算设备
#   二、数据集            —— HotpotQA 样本数 / 文件路径 / 预览下标
#   三、索引构建与缓存    —— 输出路径、缓存重建开关、索引调度、图切分
#   四、语义分配（核心）  —— 方法选择 + FFT 专用 + Anchor 专用 + 共用(Wikidata/LLM/描述)
#   五、调试与浏览        —— 短语审计、多义项浏览
# #############################################################################


# =============================================================================
# 一、运行环境
# =============================================================================
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"  # 优先使用 CUDA，没有 GPU 时回退到 CPU


# =============================================================================
# 二、数据集（HotpotQA）
# =============================================================================
NUM_SAMPLES = 500  # 使用的 HotpotQA 样本数；设为 None 则索引完整 distractor dev 集
HOTPOT_FILE_CANDIDATES = [  # HotpotQA 数据文件候选路径，按顺序取第一个存在的文件
    REPO_ROOT / "jupyter_notebooks" / "hotpot_dev_distractor_v1.json",
    REPO_ROOT / "hotpot_dev_distractor_v1.json",
]
PREVIEW_INDEX = 0  # 预览样本时使用的样本下标


# =============================================================================
# 三、索引构建与缓存
# =============================================================================
# --- 输出路径 ---
INDEX_OUTPUT_DIR = REPO_ROOT / "cache" / "hotpotqa_latest_framework_index"  # 索引与缓存文件的输出目录
INDEX_RUN_TAG = "local"  # 缓存版本标签：会拼进所有缓存文件名，用于区分不同 LLM 后端/实验配置，避免互相覆盖。例如 API LLM 用 "api"、本地 LLM 用 "local"；设为 None 或 "" 则不加标签（与旧缓存同名）

# --- 缓存 / 重建开关 ---
SAVE_SPLIT_INDEX = False  # 是否分文件保存索引与 tensor（save/load_data_split）
LOAD_SAVED_INDEX_IF_EXISTS = False  # 若已有缓存索引，是否直接加载而跳过重建
FORCE_REBUILD_INDEX = True  # 是否强制重建索引并覆盖已有缓存
VERIFY_LOAD_AFTER_SAVE = False  # 重建后是否重新加载索引，验证保存是否成功

# --- 索引调度 / 图切分 ---
GRAPH_BATCH_SIZE = 64  # 并行索引时每批处理的 chunk 数量
GRAPH_QUEUE_SIZE = 16  # 索引任务队列长度
GRAPH_CHUNK_SIZE = 256  # 文档切 chunk 时的最大字符数
MIN_OCCURRENCES_FOR_DESCRIPTION = 50  # token/phrase 至少出现多少次才进入语义描述分配流程


# =============================================================================
# 四、语义分配（核心）
# =============================================================================
# -----------------------------------------------------------------------------
# 4.1 方法选择（决定走哪条路径，下面 4.2/4.3 二选一生效）
# -----------------------------------------------------------------------------
SEMANTIC_ASSIGNMENT_METHOD = "Anchor-E-mutual [ce_fallback]"  # 义项选择方式："FFT-CE" 用 cross-encoder，"FFT-LLM" 用 LLM，"Anchor-*" 用 anchor 标注+kNN 传播

# -----------------------------------------------------------------------------
# 4.2 仅 FFT-CE / FFT-LLM 路径使用的参数
# -----------------------------------------------------------------------------
FFT_MAX_SAMPLES = 10  # farthest-first 采样时最多抽取的上下文句数
FFT_CONSENSUS_RATIO_THRESHOLD = 0.9  # 候选义项得票率需达到此阈值才认定为共识义项
FFT_D1_D2_RATIO_THRESHOLD = 0.8  # 最高票与次高票得分比低于此值时，认为义项仍有歧义
FFT_KNN_CHECK_K = 5  # KNN 检查时考察的最近邻数量
FFT_DANGER_NEIGHBOR_M = 10  # 判断 embedding 是否为"危险邻居"时参考的邻居数

# -----------------------------------------------------------------------------
# 4.3 仅 Anchor-* 路径使用的参数
# -----------------------------------------------------------------------------
ANCHOR_FRACTION = 0.15  # anchor 占某 token 全部 occurrence 的比例
ANCHOR_FFT_RATIO = 0.70  # anchor 中由 FFT 选出的比例（其余随机抽取）
ANCHOR_MIN_COUNT = 2  # 每个 token 至少选取的 anchor 个数
ANCHOR_MAX_COUNT = 15  # 每个 token 最多选取的 anchor 个数（封顶，避免高频词把过多样本塞进单次 LLM 判定）
PROP_KNN_K = 8  # kNN 传播构图时的近邻数 k

# -----------------------------------------------------------------------------
# 4.4 两条路径共用的参数（Wikidata 候选 / LLM 过滤 / 描述生成）
# -----------------------------------------------------------------------------
WIKIDATA_CANDIDATE_LIMIT = 5  # 每个 token 从 Wikidata 拉取的候选义项上限
USE_LLM_WIKIDATA = True  # 是否用 LLM 过滤 Wikidata 候选义项
LLM_WIKIDATA_USE_API = False  # Wikidata 义项过滤是否走 API（False 则用本地模型）
SEPARATE_LLM_CACHE_BY_BACKEND = True  # 是否按 LLM 后端(API/本地)分离应答缓存。注意：sqlite 缓存只按词条(term)做 key，默认会跨后端复用，开启此项后 API/本地各用独立缓存文件，避免本地 LLM 直接吃 API 的旧应答
SEM_DESCRIPTION_BATCH_SIZE = 32  # 批量打语义描述时的 batch 大小
SEM_DESCRIPTION_PROMPT_CONTEXT_MODE = "sentence_neighbors"  # 构造语义描述 prompt 时使用的上下文模式
TAU_CONC = 0.88  # s_mean 闸门:某 token 各次出现 embedding 的平均相似度高于此值即判为单义(不切分);越大越倾向多义切分

# -----------------------------------------------------------------------------
# 4.5 方法名校验与派生（自动推导，一般无需改动）
# -----------------------------------------------------------------------------
# 校验 SEMANTIC_ASSIGNMENT_METHOD 取值，并推导是否启用 LLM 语义标注器
# 支持的语义分配方式：
#   - "FFT-CE"  ：FFT 采样 + cross-encoder 选义项（不走 LLM 标注）
#   - "FFT-LLM" ：FFT 采样 + LLM 选义项
#   - Anchor-*  ：anchor 标注 + kNN 传播（RAG_graph.DEFAULT_SEM_ASSIGNMENT_METHOD 即属此类）
#     合法 anchor 名形如 "Anchor-F-mutual [ce_fallback]"：
#       版本 C/D 不带 plain/mutual 后缀；E/F 必须带 plain 或 mutual；
#       方括号内为 uncertain 模式，取 ce_fallback 或 re_llm。
_ANCHOR_VERSIONS = {"C": (None,), "D": (None,), "E": ("plain", "mutual"), "F": ("plain", "mutual")}  # 各 anchor 版本允许的 kNN 后缀
_ANCHOR_UNCERTAIN_MODES = ("ce_fallback", "re_llm")  # anchor 不确定样本的兜底模式
SUPPORTED_ANCHOR_ASSIGNMENT_METHODS = {
    (f"Anchor-{version}-{knn} [{mode}]" if knn else f"Anchor-{version} [{mode}]")
    for version, knns in _ANCHOR_VERSIONS.items()
    for knn in knns
    for mode in _ANCHOR_UNCERTAIN_MODES
}  # 枚举出的全部合法 anchor 方法名
SUPPORTED_SEMANTIC_ASSIGNMENT_METHODS = {"FFT-CE", "FFT-LLM"} | SUPPORTED_ANCHOR_ASSIGNMENT_METHODS  # 全部受支持的语义分配方式
if SEMANTIC_ASSIGNMENT_METHOD not in SUPPORTED_SEMANTIC_ASSIGNMENT_METHODS:
    raise ValueError(
        f"Unsupported SEMANTIC_ASSIGNMENT_METHOD={SEMANTIC_ASSIGNMENT_METHOD!r}; "
        f"use one of {sorted(SUPPORTED_SEMANTIC_ASSIGNMENT_METHODS)}"
    )
USE_LLM_SEMANTIC_LABELER = SEMANTIC_ASSIGNMENT_METHOD == "FFT-LLM"  # 仅 FFT-LLM 时启用 LLM 语义标注器


# =============================================================================
# 五、调试与浏览
# =============================================================================
# -----------------------------------------------------------------------------
# 5.1 短语审计（phrase audit）
# -----------------------------------------------------------------------------
ENABLE_PHRASE_AUDIT = True  # 索引时是否记录短语抽取审计日志
CLEAR_PHRASE_AUDIT_CACHE_ON_REBUILD = True  # 重建索引时是否删除旧的 phrase audit 缓存
PHRASE_AUDIT_MAX_ROWS = 10000  # 导出审计表时最多显示的行数
PHRASE_AUDIT_FILTER_PROTECTED = None  # 按 protected 字段过滤；True / False / None 表示不过滤
PHRASE_AUDIT_FILTER_REASON = None  # 按 protected_reason 过滤，例如 "TITLE_LIKE_NOUN_CHUNK"
PHRASE_AUDIT_FILTER_TEXT = None  # 按 phrase 或 raw_text 的子串过滤（不区分大小写）

# -----------------------------------------------------------------------------
# 5.2 多义项浏览
# -----------------------------------------------------------------------------
MULTI_SEM_MIN_SEM_COUNT = 2  # 只展示至少有这么多 sem node 的 token
MULTI_SEM_MAX_TOKEN_NODES = 50  # 最多展示多少个 token node
MULTI_SEM_MAX_SEMS_PER_TOKEN = 10  # 每个 token 最多展示多少个 sem node
MULTI_SEM_MAX_SENTENCES_PER_SEM = 3  # 每个 sem node 最多展示多少条例句
MULTI_SEM_MAX_EXAMPLES_PER_TOKEN = 10  # 每个 token 最多展示多少条 chunk 示例
MULTI_SEM_TOKEN_CONTAINS = None  # 只展示 surface form 包含该子串的 token；None 表示不过滤

# -----------------------------------------------------------------------------
# 5.3 Anchor LLM 样本分配记录浏览
# -----------------------------------------------------------------------------
LLM_ANCHOR_MAX_TOKEN_NODES = 50  # 最多展示多少个有 LLM anchor 记录的 token node
LLM_ANCHOR_MAX_RECORDS_PER_TOKEN = 20  # 每个 token 最多展示多少条 LLM 样本记录
LLM_ANCHOR_TOKEN_CONTAINS = None  # 只展示 surface form 包含该子串的 token；None 表示不过滤
LLM_ANCHOR_SORT_BY = "record_count"  # 可选: "record_count" / "accepted_count" / "sem_count" / "token_text"

print(f"Working directory: {REPO_ROOT}")


Working directory: /home/xiaoyue/LiteSemRAG


In [2]:
import RAG_graph

DATASET_TAG = "all" if NUM_SAMPLES is None else str(NUM_SAMPLES)
INDEX_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

INDEX_BASENAME = f"litesemrag_hotpotqa_{DATASET_TAG}"
if INDEX_RUN_TAG:  # 追加版本标签，避免不同 LLM 后端/配置的缓存互相覆盖
    INDEX_BASENAME = f"{INDEX_BASENAME}_{INDEX_RUN_TAG}"
INDEX_PKL_PATH = INDEX_OUTPUT_DIR / f"{INDEX_BASENAME}.pkl"
INDEX_TENSOR_PATH = Path(str(INDEX_PKL_PATH).replace(".pkl", "_tensors.pt"))
INDEX_DOCS_JSON_PATH = INDEX_OUTPUT_DIR / f"{INDEX_BASENAME}_documents.json"
INDEX_META_PATH = INDEX_OUTPUT_DIR / f"{INDEX_BASENAME}_meta.json"  # 记录本次索引生成参数的伴生文档

PHRASE_AUDIT_CACHE_PATH = INDEX_OUTPUT_DIR / f"{INDEX_BASENAME}_phrase_audit.jsonl"
PHRASE_AUDIT_SUMMARY_CSV_PATH = INDEX_OUTPUT_DIR / f"{INDEX_BASENAME}_phrase_audit_summary.csv"
PHRASE_AUDIT_ROWS_CSV_PATH = INDEX_OUTPUT_DIR / f"{INDEX_BASENAME}_phrase_audit_rows.csv"
PHRASE_AUDIT_REPORT_PATH = INDEX_OUTPUT_DIR / f"{INDEX_BASENAME}_phrase_audit_report.txt"

# --- LLM 应答缓存（sqlite）：按后端(API/本地)分离 ---
# sqlite 缓存只按词条(term)做 key，跨后端复用；开启 SEPARATE_LLM_CACHE_BY_BACKEND
# 后，API 用 *_api.sqlite3、本地用 *_local.sqlite3，避免本地 LLM 直接吃 API 的旧应答。
LLM_CACHE_DIR = REPO_ROOT / "cache"
LLM_BACKEND_TAG = "api" if LLM_WIKIDATA_USE_API else "local"

def _backend_cache_path(filename):
    path = LLM_CACHE_DIR / filename
    if SEPARATE_LLM_CACHE_BY_BACKEND:
        path = path.with_name(f"{path.stem}_{LLM_BACKEND_TAG}{path.suffix}")
    return path

LLM_CANDIDATE_FILTER_CACHE_PATH = _backend_cache_path("wikidata_definition_filter_cache.sqlite3")  # Wikidata 义项过滤缓存（Anchor/FFT 共用）
LLM_SEMANTIC_LABELER_CACHE_PATH = _backend_cache_path("llm_semantic_label_cache.sqlite3")  # FFT-LLM 语义标注缓存

# 解析方法名：anchor 方法返回 spec dict，FFT-CE/FFT-LLM 返回 None（走 FFT 路径）
ANCHOR_METHOD_SPEC = RAG_graph._parse_anchor_method(SEMANTIC_ASSIGNMENT_METHOD)
IS_ANCHOR_METHOD = ANCHOR_METHOD_SPEC is not None

GRAPH_CONFIG = dict(
    min_occurrences_for_description=MIN_OCCURRENCES_FOR_DESCRIPTION,
    chunk_size=GRAPH_CHUNK_SIZE,
    device=DEVICE,
    sem_assignment_method=SEMANTIC_ASSIGNMENT_METHOD,
    consensus_ratio_threshold=FFT_CONSENSUS_RATIO_THRESHOLD,
    fft_max_samples=FFT_MAX_SAMPLES,
    fft_d1_d2_ratio_threshold=FFT_D1_D2_RATIO_THRESHOLD,
    fft_knn_check_k=FFT_KNN_CHECK_K,
    fft_danger_neighbor_m=FFT_DANGER_NEIGHBOR_M,
    anchor_fraction=ANCHOR_FRACTION,
    anchor_fft_ratio=ANCHOR_FFT_RATIO,
    anchor_min_count=ANCHOR_MIN_COUNT,
    anchor_max_count=ANCHOR_MAX_COUNT,
    prop_knn_k=PROP_KNN_K,
    tau_conc=TAU_CONC,
    use_llm_candidate_filter=USE_LLM_WIKIDATA,
    use_llm_semantic_labeler=USE_LLM_SEMANTIC_LABELER,
    llm_candidate_filter_use_api=LLM_WIKIDATA_USE_API,
    llm_candidate_filter_cache_path=str(LLM_CANDIDATE_FILTER_CACHE_PATH),
    llm_semantic_labeler_cache_path=str(LLM_SEMANTIC_LABELER_CACHE_PATH),
    sem_description_prompt_context_mode=SEM_DESCRIPTION_PROMPT_CONTEXT_MODE,
    phrase_audit_enabled=ENABLE_PHRASE_AUDIT,
    phrase_audit_cache_path=str(PHRASE_AUDIT_CACHE_PATH),
)


def apply_semantic_assignment_runtime_settings(graph):
    graph.sem_description_candidate_limit = WIKIDATA_CANDIDATE_LIMIT
    graph.llm_candidate_filter_candidate_limit = WIKIDATA_CANDIDATE_LIMIT
    graph.use_llm_semantic_labeler = USE_LLM_SEMANTIC_LABELER
    graph.sem_description_batch_size = SEM_DESCRIPTION_BATCH_SIZE
    graph.sem_description_prompt_context_mode = SEM_DESCRIPTION_PROMPT_CONTEXT_MODE
    return graph


def write_index_metadata(graph=None, path=INDEX_META_PATH):
    """把本次索引生成时使用的全部关键参数落盘成 JSON 伴生文档，便于区分/复现每个缓存文件。"""
    import json
    from datetime import datetime

    meta = {
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "index_run_tag": INDEX_RUN_TAG,
        "index_basename": INDEX_BASENAME,
        "dataset": {
            "num_samples": NUM_SAMPLES,
            "dataset_tag": DATASET_TAG,
        },
        "llm_backend": {
            "use_llm_wikidata": USE_LLM_WIKIDATA,
            "llm_wikidata_use_api": LLM_WIKIDATA_USE_API,  # True=API LLM, False=本地 LLM
            "use_llm_semantic_labeler": USE_LLM_SEMANTIC_LABELER,
            "wikidata_candidate_limit": WIKIDATA_CANDIDATE_LIMIT,
            "sem_description_batch_size": SEM_DESCRIPTION_BATCH_SIZE,
            "separate_llm_cache_by_backend": SEPARATE_LLM_CACHE_BY_BACKEND,
            "llm_backend_tag": LLM_BACKEND_TAG,
            "llm_candidate_filter_cache_path": str(LLM_CANDIDATE_FILTER_CACHE_PATH),
            "llm_semantic_labeler_cache_path": str(LLM_SEMANTIC_LABELER_CACHE_PATH),
        },
        "semantic_assignment_method": SEMANTIC_ASSIGNMENT_METHOD,
        # GRAPH_CONFIG 已涵盖切分/FFT/Anchor/tau_conc 等全部构图参数
        "graph_config": {k: (str(v) if isinstance(v, Path) else v) for k, v in GRAPH_CONFIG.items()},
        "output_paths": {
            "index_pkl": str(INDEX_PKL_PATH),
            "index_tensor": str(INDEX_TENSOR_PATH) if SAVE_SPLIT_INDEX else None,
            "documents_json": str(INDEX_DOCS_JSON_PATH),
            "phrase_audit_jsonl": str(PHRASE_AUDIT_CACHE_PATH),
        },
    }
    if graph is not None:
        meta["graph_stats"] = {
            "docs": len(graph.doc_nodes),
            "chunks": len(graph.chunk_nodes),
            "tokens": len(graph.token_nodes),
            "phrase_tokens": len(graph.phrase_token_nodes),
            "sem_nodes": len(graph.sem_nodes),
        }
    Path(path).write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    print(f"Saved index metadata to: {path}")
    return meta


print(f"Device: {DEVICE}")
print(f"Index run tag: {INDEX_RUN_TAG}")
print(f"Index pickle path: {INDEX_PKL_PATH}")
print(f"Index tensor path: {INDEX_TENSOR_PATH}")
print(f"Index document JSON path: {INDEX_DOCS_JSON_PATH}")
print(f"Index metadata path: {INDEX_META_PATH}")
print(f"LLM backend tag: {LLM_BACKEND_TAG} (separate cache: {SEPARATE_LLM_CACHE_BY_BACKEND})")
print(f"LLM candidate filter cache path: {LLM_CANDIDATE_FILTER_CACHE_PATH}")
print(f"LLM semantic labeler cache path: {LLM_SEMANTIC_LABELER_CACHE_PATH}")
print(f"Semantic assignment method: {SEMANTIC_ASSIGNMENT_METHOD}")

# 只打印当前语义分配方式实际用到的参数，避免无关参数造成误导
if IS_ANCHOR_METHOD:
    print(f"  Anchor version: {ANCHOR_METHOD_SPEC['version']}")
    print(f"  Anchor kNN type: {ANCHOR_METHOD_SPEC['knn_type']}")
    print(f"  Anchor uncertain mode: {ANCHOR_METHOD_SPEC['uncertain_mode']}")
    print(f"  Anchor fraction: {ANCHOR_FRACTION}")
    print(f"  Anchor FFT ratio: {ANCHOR_FFT_RATIO}")
    print(f"  Anchor min count: {ANCHOR_MIN_COUNT}")
    print(f"  Anchor max count: {ANCHOR_MAX_COUNT}")
    print(f"  Propagation kNN k: {PROP_KNN_K}")
else:
    print(f"  Use LLM semantic labeler: {USE_LLM_SEMANTIC_LABELER}")
    print(f"  FFT max samples: {FFT_MAX_SAMPLES}")
    print(f"  FFT consensus ratio threshold: {FFT_CONSENSUS_RATIO_THRESHOLD}")
    print(f"  FFT d1/d2 ratio threshold: {FFT_D1_D2_RATIO_THRESHOLD}")
    print(f"  FFT KNN check k: {FFT_KNN_CHECK_K}")
    print(f"  FFT danger neighbor m: {FFT_DANGER_NEIGHBOR_M}")

# 两条路径共用的参数
print(f"Use LLM Wikidata filter: {USE_LLM_WIKIDATA}")
print(f"Wikidata candidate limit: {WIKIDATA_CANDIDATE_LIMIT}")
print(f"Sem description batch size: {SEM_DESCRIPTION_BATCH_SIZE}")
print(f"Sem description prompt context mode: {SEM_DESCRIPTION_PROMPT_CONTEXT_MODE}")
print(f"Min occurrences for description: {MIN_OCCURRENCES_FOR_DESCRIPTION}")
print(f"s_mean concentration gate (tau_conc): {TAU_CONC}")
print(f"Phrase audit enabled: {ENABLE_PHRASE_AUDIT}")
print(f"Phrase audit cache path: {PHRASE_AUDIT_CACHE_PATH}")
print(f"Load saved index if available: {LOAD_SAVED_INDEX_IF_EXISTS}")
print(f"Force rebuild index: {FORCE_REBUILD_INDEX}")

Device: cuda
Index run tag: local
Index pickle path: /home/xiaoyue/LiteSemRAG/cache/hotpotqa_latest_framework_index/litesemrag_hotpotqa_500_local.pkl
Index tensor path: /home/xiaoyue/LiteSemRAG/cache/hotpotqa_latest_framework_index/litesemrag_hotpotqa_500_local_tensors.pt
Index document JSON path: /home/xiaoyue/LiteSemRAG/cache/hotpotqa_latest_framework_index/litesemrag_hotpotqa_500_local_documents.json
Index metadata path: /home/xiaoyue/LiteSemRAG/cache/hotpotqa_latest_framework_index/litesemrag_hotpotqa_500_local_meta.json
LLM backend tag: local (separate cache: True)
LLM candidate filter cache path: /home/xiaoyue/LiteSemRAG/cache/wikidata_definition_filter_cache_local.sqlite3
LLM semantic labeler cache path: /home/xiaoyue/LiteSemRAG/cache/llm_semantic_label_cache_local.sqlite3
Semantic assignment method: Anchor-E-mutual [ce_fallback]
  Anchor version: E
  Anchor kNN type: mutual
  Anchor uncertain mode: ce_fallback
  Anchor fraction: 0.15
  Anchor FFT ratio: 0.7
  Anchor min count: 

In [3]:
from text_processing import *
from utils import *

file_path = next(
    (str(path) for path in HOTPOT_FILE_CANDIDATES if path.exists()),
    str(HOTPOT_FILE_CANDIDATES[0]),
)
print(f"Using HotpotQA file: {file_path}")

documents, samples = build_hotpot_retrieval_dataset(file_path, num_samples=NUM_SAMPLES)

print(f"Documents: {len(documents)}")
print(f"Samples: {len(samples)}")


Using HotpotQA file: /home/xiaoyue/LiteSemRAG/jupyter_notebooks/hotpot_dev_distractor_v1.json
Loading cached dataset...
Loaded 4937 documents
Loaded 500 samples
Documents: 4937
Samples: 500


In [4]:
print("Question:", samples[PREVIEW_INDEX]["question"])
print("Answer:", samples[PREVIEW_INDEX]["answer"])
print("Gold doc ids:", samples[PREVIEW_INDEX]["gold_doc_ids"])

for doc_id in samples[PREVIEW_INDEX]["gold_doc_ids"]:
    doc = documents[doc_id]
    print("\nTitle:", doc["title"])
    print(doc["text"][:500])


Question: Were Scott Derrickson and Ed Wood of the same nationality?
Answer: yes
Gold doc ids: [1, 4]

Title: Scott Derrickson
Scott Derrickson (born July 16, 1966) is an American director, screenwriter and producer.  He lives in Los Angeles, California.  He is best known for directing horror films such as "Sinister", "The Exorcism of Emily Rose", and "Deliver Us From Evil", as well as the 2016 Marvel Cinematic Universe installment, "Doctor Strange."

Title: Ed Wood
Edward Davis Wood Jr. (October 10, 1924 – December 10, 1978) was an American filmmaker, actor, writer, producer, and director.


In [5]:
def index_cache_exists():
    if SAVE_SPLIT_INDEX:
        return INDEX_PKL_PATH.exists() and INDEX_TENSOR_PATH.exists()
    return INDEX_PKL_PATH.exists()

INDEX_WAS_REBUILT = False
INDEX_CACHE_AVAILABLE = index_cache_exists()

if LOAD_SAVED_INDEX_IF_EXISTS and INDEX_CACHE_AVAILABLE and not FORCE_REBUILD_INDEX:
    print(f"Loading saved index from: {INDEX_PKL_PATH}")
    if SAVE_SPLIT_INDEX:
        graph_database = RAG_graph.LiteSemRAG.load_data_split(str(INDEX_PKL_PATH))
    else:
        graph_database = RAG_graph.LiteSemRAG.load_data(str(INDEX_PKL_PATH))
    graph_database.json_path = str(INDEX_DOCS_JSON_PATH)
    apply_semantic_assignment_runtime_settings(graph_database)
    if ENABLE_PHRASE_AUDIT:
        graph_database.enable_phrase_audit(str(PHRASE_AUDIT_CACHE_PATH))
else:
    if FORCE_REBUILD_INDEX and INDEX_CACHE_AVAILABLE:
        print("Force rebuild enabled; existing cache will be overwritten after indexing.")
    elif LOAD_SAVED_INDEX_IF_EXISTS:
        print("No complete saved index cache found; building a new index.")
    else:
        print("Saved-index loading disabled; building a new index.")

    if ENABLE_PHRASE_AUDIT and CLEAR_PHRASE_AUDIT_CACHE_ON_REBUILD and PHRASE_AUDIT_CACHE_PATH.exists():
        PHRASE_AUDIT_CACHE_PATH.unlink()
        print(f"Removed old phrase audit cache: {PHRASE_AUDIT_CACHE_PATH}")

    graph_database = RAG_graph.LiteSemRAG(**GRAPH_CONFIG)
    graph_database.json_path = str(INDEX_DOCS_JSON_PATH)
    apply_semantic_assignment_runtime_settings(graph_database)

    graph_database.index_json(
        documents,
        batch_size=GRAPH_BATCH_SIZE,
        queue_size=GRAPH_QUEUE_SIZE,
        sample_count=len(samples),
    )
    graph_database.finalize()
    graph_database.print_memory_size()

    if SAVE_SPLIT_INDEX:
        graph_database.save_data_split(str(INDEX_PKL_PATH))
        print(f"Saved split index to: {INDEX_PKL_PATH}")
        print(f"Saved tensor file to: {INDEX_TENSOR_PATH}")
    else:
        graph_database.save_data(str(INDEX_PKL_PATH))
        print(f"Saved index to: {INDEX_PKL_PATH}")

    # 把本次构建参数写入伴生 JSON，和 .pkl 一一对应，便于事后区分/复现
    write_index_metadata(graph_database)

    INDEX_WAS_REBUILT = True

print(f"Saved indexed document list path: {INDEX_DOCS_JSON_PATH}")
if ENABLE_PHRASE_AUDIT:
    print(f"Phrase audit cache path: {PHRASE_AUDIT_CACHE_PATH}")
print("Graph stats:")
print(f"  loaded from cache: {not INDEX_WAS_REBUILT}")
print(f"  docs: {len(graph_database.doc_nodes)}")
print(f"  chunks: {len(graph_database.chunk_nodes)}")
print(f"  tokens: {len(graph_database.token_nodes)}")
print(f"  phrase tokens: {len(graph_database.phrase_token_nodes)}")
print(f"  sem nodes: {len(graph_database.sem_nodes)}")


Saved-index loading disabled; building a new index.
Loading text encoder models in device: GPU


/home/xiaoyue/anaconda3/envs/llm_graph/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


CPU preprocessed: 4937/4937 | GPU encoded: 4937/4937 | CPU processed: 4937/4937
[1081.2983s] Index 4937 documents. Index pipeline time: 1081.2983s
[1081.3014s] Finalize started.
[1081.3069s] Computed average chunk length.
[1081.5038s] Removed 3344 empty placeholder token nodes.
finalize_token_nodes: 450 potential multi-sense node(s) (occurrences >= min_occurrences_for_description=50), 58655 basic node(s).
  Building potential multi-sense nodes (most time-consuming stage)...
multi_sense_node_build: 450/450 | remaining:   0
basic_node_build: 58655/58655 | remaining:     0
[6118.9747s] Finished token node finalization.
sem-build funnel (min_occurrences_for_description=50):
  1. 通过出现次数闸门进入完整建节点路径: 450 (其中 entity/原子短语默认单义: 114)
  2. 参与 S_mean 判定(非 entity): 336 -> 因 S_mean 过高被筛为单义: 58
  3. 通过 S_mean 进入消歧路径: 278 -> 最终切出多个语义: 90
[6119.6256s] Finished building modifier postings.
[6119.7722s] Finished assigning token and sem IDF.
[6120.0955s] Finished computing sem BM25.
[6120.4196s] Finished bu

In [6]:
import json
import pandas as pd

phrase_audit_files = sorted(INDEX_OUTPUT_DIR.glob("*phrase_audit*.jsonl"))


phrase_audit_warnings = []


def load_phrase_audit(path=PHRASE_AUDIT_CACHE_PATH):
    path = Path(path)
    phrase_audit_warnings.clear()
    if not path.exists():
        phrase_audit_warnings.append(f"Phrase audit cache not found: {path}")
        return pd.DataFrame()

    records = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                phrase_audit_warnings.append(f"Skipping invalid JSONL line {line_number}: {exc}")
    return pd.DataFrame(records)


phrase_audit_df = load_phrase_audit()

summary_cols = ["source", "protected", "protected_reason", "entity_label"]
if phrase_audit_df.empty:
    summary_df = pd.DataFrame(columns=summary_cols + ["count"])
    browse_df = phrase_audit_df.copy()
else:
    summary_df = (
        phrase_audit_df
        .groupby(summary_cols, dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
    )

    browse_df = phrase_audit_df.copy()
    if PHRASE_AUDIT_FILTER_PROTECTED is not None:
        browse_df = browse_df[browse_df["protected"] == PHRASE_AUDIT_FILTER_PROTECTED]
    if PHRASE_AUDIT_FILTER_REASON:
        browse_df = browse_df[browse_df["protected_reason"] == PHRASE_AUDIT_FILTER_REASON]
    if PHRASE_AUDIT_FILTER_TEXT:
        pattern = str(PHRASE_AUDIT_FILTER_TEXT)
        text_mask = (
            browse_df["phrase"].fillna("").str.contains(pattern, case=False, regex=False)
            | browse_df["raw_text"].fillna("").str.contains(pattern, case=False, regex=False)
        )
        browse_df = browse_df[text_mask]

display_cols = [
    "doc_name",
    "chunk_index",
    "chunk_node_id",
    "phrase",
    "raw_text",
    "source",
    "protected",
    "protected_reason",
    "entity_label",
    "start_char",
    "end_char",
]
available_display_cols = [col for col in display_cols if col in browse_df.columns]
rows_df = browse_df[available_display_cols].head(PHRASE_AUDIT_MAX_ROWS).copy()

summary_df.to_csv(PHRASE_AUDIT_SUMMARY_CSV_PATH, index=False)
rows_df.to_csv(PHRASE_AUDIT_ROWS_CSV_PATH, index=False)

report_lines = [
    f"Loaded phrase audit rows: {len(phrase_audit_df):,}",
    f"Saved summary CSV: {PHRASE_AUDIT_SUMMARY_CSV_PATH}",
    f"Saved rows CSV: {PHRASE_AUDIT_ROWS_CSV_PATH}",
    "Available phrase audit caches:",
]
if phrase_audit_files:
    report_lines.extend(f"  {path} ({path.stat().st_size:,} bytes)" for path in phrase_audit_files)
else:
    report_lines.append("  (none)")
if phrase_audit_warnings:
    report_lines.append("Warnings:")
    report_lines.extend(f"  {warning}" for warning in phrase_audit_warnings)
PHRASE_AUDIT_REPORT_PATH.write_text("\n".join(report_lines) + "\n", encoding="utf-8")
phrase_audit_output_paths = {
    "summary_csv": PHRASE_AUDIT_SUMMARY_CSV_PATH,
    "rows_csv": PHRASE_AUDIT_ROWS_CSV_PATH,
    "report_txt": PHRASE_AUDIT_REPORT_PATH,
}


In [7]:
from IPython.display import display

multi_sem_token_nodes = [
    token_node
    for token_node in graph_database.token_nodes
    if len(token_node.sem_node_list) >= MULTI_SEM_MIN_SEM_COUNT
]

print(
    f"Token nodes with >= {MULTI_SEM_MIN_SEM_COUNT} sem nodes: "
    f"{len(multi_sem_token_nodes)} / {len(graph_database.token_nodes)}"
)

display(
    graph_database.show_multi_sem_token_nodes(
        min_sem_count=MULTI_SEM_MIN_SEM_COUNT,
        max_sentences_per_sem=MULTI_SEM_MAX_SENTENCES_PER_SEM,
        as_html=True,
        token_contains=MULTI_SEM_TOKEN_CONTAINS,
        sort_by="sem_count",
        max_token_nodes=MULTI_SEM_MAX_TOKEN_NODES,
        max_sems_per_token=MULTI_SEM_MAX_SEMS_PER_TOKEN,
        max_examples_per_token=MULTI_SEM_MAX_EXAMPLES_PER_TOKEN,
        open_details=False,
    )
)


Token nodes with >= 2 sem nodes: 90 / 59105


In [8]:
# 不在 notebook 中渲染海量嵌套 <details>(会卡死页面),改为把完整的 Anchor LLM
# 样本分配记录(含 matched_text / context_text / span_text 原文)展平成 CSV 存到本地 logs/。
# 落盘是完整导出(不做 max_token_nodes / max_records_per_token 截断),仅保留可选的
# token_contains 过滤与排序,确保表格里不丢记录。
records_by_token = graph_database.inspect_llm_anchor_sample_assignments()
total_records = sum(info["record_count"] for info in records_by_token.values())
print(
    "LLM Anchor sample assignment records: "
    f"{total_records} records across {len(records_by_token)} token nodes"
)

llm_anchor_csv_path = graph_database.save_llm_anchor_sample_assignments_csv(
    token_contains=LLM_ANCHOR_TOKEN_CONTAINS,
    sort_by=LLM_ANCHOR_SORT_BY,
    max_token_nodes=None,
    max_records_per_token=None,
)


LLM Anchor sample assignment records: 3587 records across 278 token nodes
Saved 3587 LLM Anchor sample assignment records (278 token nodes) to /home/xiaoyue/LiteSemRAG/logs/llm_anchor_assignments_20260605_230828.csv


In [9]:
if VERIFY_LOAD_AFTER_SAVE:
    if INDEX_WAS_REBUILT:
        if SAVE_SPLIT_INDEX:
            loaded_graph = RAG_graph.LiteSemRAG.load_data_split(str(INDEX_PKL_PATH))
        else:
            loaded_graph = RAG_graph.LiteSemRAG.load_data(str(INDEX_PKL_PATH))
        print("Reloaded graph from saved index for verification.")
    else:
        loaded_graph = graph_database
        print("Using graph already loaded from saved index; reload verification skipped.")

    print("Loaded graph stats:")
    print(f"  docs: {len(loaded_graph.doc_nodes)}")
    print(f"  chunks: {len(loaded_graph.chunk_nodes)}")
    print(f"  tokens: {len(loaded_graph.token_nodes)}")
    print(f"  phrase tokens: {len(loaded_graph.phrase_token_nodes)}")
    print(f"  sem nodes: {len(loaded_graph.sem_nodes)}")
    print(f"  query database shape: {None if loaded_graph.query_database is None else tuple(loaded_graph.query_database.shape)}")
